<a href="https://colab.research.google.com/github/Diagnost13/simulative-git/blob/master/%D0%92%D1%82%D0%BE%D1%80%D0%BE%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2 кейс

**Соберите данные из Google Sheets и напишите функцию `generate_report`, которая принимает три списка (данные с каждого листа в Google Sheets) и сохраняет отчет в файл `student_debt_report.txt`**

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [37]:
!wget https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json

!wget https://gist.github.com/Vs8th/39c5deed0f5539d781f00328f7fd4fe0/raw/result.txt

--2026-03-05 17:55:22--  https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.github.com (gist.github.com)... 20.27.177.113
Connecting to gist.github.com (gist.github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json [following]
--2026-03-05 17:55:22--  https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2358 (2.3K) [text/plain]
Saving to: ‘creds.json.1’

creds.json.1        100%[===================>]   2.30K  --.-KB/s    in 0s      

2026-03-05 17:55:23 (44.7 MB/s) - ‘creds.json.1’ sa

Чтобы посмотреть как выглядят сами гугл таблицы в оригинале - можете перейти по [ссылке](https://docs.google.com/spreadsheets/d/1hRnw-PEftF0J-6KY7InlILVwWdkJu4vJiGwRjuE_P4U/edit?usp=sharing) и изучить.

Небольшое описание столбцов в таблицах:  
**Лист1:** `student_id` - айди студента; `student_name` - имя студента; `installment` (Y/N) - факт наличия рассрочки у студента (Y - рассрочка есть, N - рассрочки нет).  
**Лист2:** `student_id` - айди студента; `last_payment_date` - дата последней полученной оплаты; `expected_payment_date` - дата, в которую ожидаем следующий платеж (рассчитывается от даты последнего платежа).  
**Лист3:** `student_id` - айди студента; `already_payed_amount` - уже выплаченная часть рассрочки; `left_to_pay` - оставшаяся (невыплаченная) часть; `one-time_payment` - единоразовый платеж; `installment_amount` - общая сумма, которая бралась в рассрочку.

Как примерно должен выглядеть результат:

In [38]:
with open('result.txt') as f:
  print(f.read())

Студент Иванов У.У. - долг 100000 рублей
Студент Петров Е.Е. - долг 250000 рублей
Студент Сидоров И.И. - долг 10000 рублей


In [39]:
#@title Если возникнут трудности с определением `scope`, подсказка:

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

**Примечание**

Считать должников необходимо на 1 марта 2023 года. То есть определяя количество просроченных платежей, мы определяем их количество не на настоящую дату, а на **1 марта 2023 года**. А периоды внесения платежей для всех студентов одинаковы - **6 месяцев** (183 дня).

То есть, если ожидаемый платеж должен был пройти 3 февраля 2022 года, то к 1 марта 2023 студент просрочил уже три платежа (3 февраля 2022, 3 августа 2022 и 3 февраля 2023). При расчетах отталкивайтесь от даты ожидаемого платежа, разницу между платежами берите 183 дня.

**Решение**

Напишите свое решение ниже

In [40]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']
creds = ServiceAccountCredentials.from_json_keyfile_name('creds.json', scope)
client = gspread.authorize(creds)
spreadsheet = client.open_by_key("1hRnw-PEftF0J-6KY7InlILVwWdkJu4vJiGwRjuE_P4U")
sheet1_data = spreadsheet.worksheet("Лист1").get_all_records()
sheet2_data = spreadsheet.worksheet("Лист2").get_all_records()
sheet3_data = spreadsheet.worksheet("Лист3").get_all_records()

In [41]:
from datetime import datetime, timedelta

def generate_debt_report(sheet1_data, sheet2_data, sheet3_data):
    """
    Генерирует отчёт о студентах с долгами на 1 марта 2023 года.

    Args:
        sheet1_data (list): данные о студентах (из Листа1)
        sheet2_data (list): данные о платежах (из Листа2)
        sheet3_data (list): финансовые данные (из Листа3)
    """
    # Дата, на которую делаем расчёт
    REPORT_DATE = datetime(2023, 3, 1)
    PAYMENT_INTERVAL_DAYS = 183  # Период между платежами в днях (6 месяцев)

    # Создаём словарь для хранения информации о долгах
    debt_info = {}

    # Преобразуем данные в словари для удобства работы

    # Словарь студентов
    students_dict = {}
    for row in sheet1_data:
        student_id = row['student_id']
        students_dict[student_id] = {
            'name': row['student_name'],
            'has_installment': row['installment'] == 'Y'  # Y — рассрочка есть, N — рассрочки нет
        }

    # Словарь платежей
    payments_dict = {}
    for row in sheet2_data:
        student_id = row['student_id']
        payments_dict[student_id] = {
            'last_payment_date': row['last_payment_date'],
            'expected_payment_date': row['expected_payment_date']
        }

    # Финансовый словарь
    financial_dict = {}
    for row in sheet3_data:
        student_id = row['student_id']
        financial_dict[student_id] = {
            'already_payed_amount': float(row['already_payed_amount']),
            'left_to_pay': float(row['left_to_pay']),
            'one_time_payment': float(row['one_time_payment']),
            'installment_amount': float(row['installment_amount'])
        }

    # Обрабатываем только студентов с рассрочкой
    for student_id, student_info in students_dict.items():
        if not student_info['has_installment']:
            continue  # Пропускаем студентов без рассрочки

        # Проверяем, есть ли данные о платежах и финансах для этого студента
        if student_id not in payments_dict or student_id not in financial_dict:
            continue

        payment_info = payments_dict[student_id]
        finance_info = financial_dict[student_id]

        # Парсим дату ожидаемого платежа
        expected_date_str = payment_info['expected_payment_date']
        expected_date = datetime.strptime(expected_date_str, '%Y-%m-%d')

        # Если ожидаемый платёж уже прошёл на дату отчёта — студент в просрочке
        if expected_date <= REPORT_DATE:
            # Считаем количество пропущенных платежей на 01.03.2023
            missed_payments = 0
            current_expected_date = expected_date

            # Цикл: добавляем платежи каждые 183 дня, пока не превысим дату отчёта
            while current_expected_date <= REPORT_DATE:
                missed_payments += 1
                current_expected_date += timedelta(days=PAYMENT_INTERVAL_DAYS)

            # Расчёт суммы долга
            if finance_info['one_time_payment'] > 0:
                one_payment_amount = finance_info['one_time_payment']
            else:
                # Если one-time_payment не задан, рассчитываем как оставшаяся сумма / количество пропущенных
                if missed_payments > 0:
                    one_payment_amount = finance_info['left_to_pay'] / missed_payments
                else:
                    one_payment_amount = 0

            total_debt = missed_payments * one_payment_amount

            debt_info[student_id] = {
                'name': student_info['name'],
                'debt': round(total_debt, 2)
            }

    # Формируем отчёт в требуемом формате
    report_lines = []
    for student_id, info in debt_info.items():
        if info['debt'] > 0:  # Только студенты с долгом
            report_lines.append(f"Студент {info['name']} - долг {int(info['debt'])} рублей")

    # Сохраняем отчёт в файл
    with open('student_debt_report.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines))

    print(f"Отчёт успешно сгенерирован. Найдено {len(report_lines)} студентов с долгами.")

# Пример использования:
# generate_debt_report(sheet1_data, sheet2_data, sheet3_data)


✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [42]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt

import pandas as pd

user_answer = pd.read_csv('student_debt_report.txt')
correct_answer = pd.read_csv('student_debt.txt')

--2026-03-05 17:55:26--  https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.github.com (gist.github.com)... 20.27.177.113
Connecting to gist.github.com (gist.github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt [following]
--2026-03-05 17:55:26--  https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11325 (11K) [text/plain]
Saving to: ‘student_debt.txt.3’

student_debt.txt.3  100%[===================>]  11.06K  --.-KB/s    in 0.001s  

2026-03-05 17:55:26 (10.5 M

In [43]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
